# Water-Quality Proxies from Satellite Imagery

## Proxies de calidad del agua a partir de imágenes satelitales

### Caso de estudio: Lago Titicaca – Bolivia

---

En este ejercicio utilizaremos imágenes satelitales para explorar algunas
propiedades relacionadas con la calidad superficial del agua en el
**Lago Titicaca**, con énfasis en el **Lago Menor o Wiñaymarca**
y la **Bahía de Cohana**, en Bolivia.

Utilizaremos dos sensores complementarios:

- **Sentinel-2 MSI**
- **MODIS Aqua**

Analizaremos cuatro variables relacionadas con la calidad del agua:

- clorofila-a;
- turbidez;
- sólidos suspendidos;
- temperatura superficial.

La pregunta que guiará el ejercicio será:

> **¿Qué información sobre la calidad superficial del agua podemos obtener
> desde el espacio y qué información complementaria aportan Sentinel-2
> y MODIS?**

A lo largo del ejercicio diferenciaremos entre:

**proxies espectrales**

y

**productos satelitales derivados**.

# 0. Introducción

La calidad del agua puede variar considerablemente dentro de un lago.

Algunos factores que pueden modificar sus propiedades ópticas son:

- fitoplancton y algas;
- sedimentos suspendidos;
- materia orgánica disuelta;
- vegetación acuática;
- profundidad del agua.

Los satélites ópticos no miden directamente la "calidad del agua".

Los sensores registran la radiación que llega hasta ellos en diferentes
regiones del espectro electromagnético.

De manera simplificada:

**Radiación solar**

**Superficie del agua**

**Absorción + dispersión**

**Señal espectral**

**Sensor satelital**

**Bandas espectrales**

**Índices o algoritmos**

**Información relacionada con calidad del agua**

---

En este ejercicio exploraremos cómo distintas características espectrales
pueden utilizarse como indicadores de condiciones superficiales del agua.

# 1. Objetivos del ejercicio

Al finalizar este ejercicio podrás:

1. Acceder a imágenes Sentinel-2 y MODIS desde Google Earth Engine.

2. Reconocer algunas bandas espectrales relevantes para el monitoreo
   de calidad del agua.

3. Crear un compuesto Sentinel-2 con baja presencia de nubes.

4. Delimitar aproximadamente la superficie de agua mediante MNDWI.

5. Calcular el **Normalized Difference Chlorophyll Index (NDCI)**
   como proxy relativo de clorofila-a.

6. Calcular el **Normalized Difference Turbidity Index (NDTI)**
   como proxy relativo de turbidez.

7. Utilizar la reflectancia en la banda roja como indicador relativo
   de material suspendido.

8. Explorar productos MODIS de:
   - clorofila-a;
   - temperatura superficial.

9. Comparar información obtenida con sensores de diferente
   resolución espacial.

10. Reconocer las principales limitaciones de los productos
    satelitales de calidad del agua.

# 2. ¿Qué podemos observar desde el espacio?

No todas las variables de calidad del agua pueden observarse directamente
mediante sensores satelitales.

En este ejercicio trabajaremos con cuatro variables.

| Variable | Información satelital utilizada |
|---|---|
| Clorofila-a | Sentinel-2 NDCI + MODIS chlor_a |
| Turbidez | Sentinel-2 NDTI |
| Sólidos suspendidos | Reflectancia roja Sentinel-2 |
| Temperatura superficial | MODIS SST |

---

## Dos conceptos importantes

### Proxy espectral

Es una variable calculada a partir de la reflectancia que puede estar
relacionada con una propiedad del agua.

Ejemplos:

- NDCI;
- NDTI;
- reflectancia roja.

Un proxy permite comparar espacialmente condiciones relativas, pero
**no representa automáticamente una concentración física**.

---

### Producto derivado

Es una variable estimada mediante un algoritmo desarrollado y procesado
por una agencia o institución.

Por ejemplo, MODIS proporciona directamente:

- `chlor_a` en mg/m³;
- `sst` en °C.

Sin embargo, incluso estos productos siguen siendo estimaciones
satelitales y presentan incertidumbre.

# 3. Datos satelitales utilizados

## Sentinel-2 MSI

Utilizaremos:

`COPERNICUS/S2_SR_HARMONIZED`

Sentinel-2 proporciona imágenes multiespectrales de alta resolución espacial.

Las principales bandas que utilizaremos son:

| Banda | Región | Resolución | Uso |
|---|---|---:|---|
| B2 | Azul | 10 m | Color natural |
| B3 | Verde | 10 m | Agua / turbidez |
| B4 | Rojo | 10 m | Turbidez / sólidos suspendidos |
| B5 | Red Edge | 20 m | Clorofila |
| B11 | SWIR | 20 m | Máscara de agua |

---

## MODIS Aqua

Utilizaremos:

`NASA/OCEANDATA/MODIS-Aqua/L3SMI`

Este producto contiene variables derivadas relacionadas con color
del océano y temperatura superficial.

Utilizaremos:

| Banda | Variable | Unidad |
|---|---|---|
| chlor_a | Clorofila-a | mg/m³ |
| sst | Temperatura superficial | °C |

La resolución espacial es de aproximadamente **4.6 km**.

---

## Pregunta

Antes de comenzar:

> ¿Qué ventajas y desventajas esperarías encontrar al comparar un sensor
> de 10–20 m con otro de aproximadamente 4.6 km?

# 4. Preparación del entorno

Utilizaremos las siguientes librerías:

- `ee`: Google Earth Engine;
- `geemap`: visualización de mapas;
- `pandas`: tablas;
- `numpy`: operaciones numéricas;
- `matplotlib`: gráficos.

In [1]:
# ============================================================
# Importar librerías
# ============================================================

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## Autenticación en Google Earth Engine

Utilizaremos el mismo procedimiento del ejercicio anterior.

Al ejecutar la siguiente celda aparecerá un enlace.

1. Abre el enlace.
2. Autoriza tu cuenta.
3. Copia el código generado.
4. Pégalo nuevamente en Colab.

In [2]:
# ============================================================
# Autenticación e inicialización de Google Earth Engine
# ============================================================

PROJECT_ID = "thermal-formula-508820-c8"

# Autenticación manual mediante enlace y código
ee.Authenticate(auth_mode="notebook")

# Inicializar Earth Engine con el proyecto de Google Cloud
ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine inicializado correctamente.")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=HTBjpETgnoj-0-HfUpx9CF9sbGiRUjy6XJqRBhFP0Rs&tc=5OY7H6qgijos0D35M1kSYCL9I_z_KPu4tgNBelqR-a0&cc=BIfVu5HuUmXxug7RVQ6KRUVLdnr_Pq43pnqar5AYF4c

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1ATsMZqDkzR8mgaHBfOi2bQCk70ZX9adsQkItcUIeKtLq9HdWgv1YK4ROveU

Successfully saved authorization token.
Google Earth Engine inicializado correctamente.


# 5. Área de estudio

Nuestro análisis se concentrará en el sector boliviano del
**Lago Menor o Wiñaymarca**.

Dentro del área se encuentra la **Bahía de Cohana**, localizada en la
ribera oriental del lago y asociada a la desembocadura del río Katari.

La zona constituye un caso interesante porque permite comparar:

- ambientes costeros;
- bahías someras;
- sectores próximos a entradas fluviales;
- aguas más abiertas del lago.

Definiremos una ventana rectangular suficientemente grande para
incluir Cohana y sectores abiertos del Lago Menor.

In [3]:
# ============================================================
# Área de estudio
# ============================================================

roi = ee.Geometry.Rectangle([
    -68.90,   # oeste
    -16.55,   # sur
    -68.55,   # este
    -16.20    # norte
])

# Punto de referencia aproximado de Cohana
cohana = ee.Geometry.Point([
    -68.660,
    -16.371
])

print("Área de estudio definida.")

Área de estudio definida.


In [4]:
# ============================================================
# Visualizar área de estudio
# ============================================================

Map = geemap.Map()

Map.centerObject(
    roi,
    10
)

Map.add_basemap(
    "HYBRID"
)

Map.addLayer(
    roi,
    {"color": "red"},
    "Área de estudio"
)

Map.addLayer(
    cohana,
    {"color": "yellow"},
    "Cohana"
)

Map

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Observa el área

Identifica:

- Bahía de Cohana;
- sectores costeros;
- islas;
- zonas abiertas del Lago Menor.

### Pregunta

> ¿Esperaríamos encontrar las mismas condiciones de calidad del agua
> dentro de una bahía costera que en aguas más abiertas?

# 6. Exploración de Sentinel-2

Trabajaremos con el período:

**1 de julio – 31 de agosto de 2025**

Esta ventana corresponde a la estación seca del Altiplano y permite
trabajar con un período relativamente corto y reciente.

En lugar de seleccionar una única imagen construiremos un
**compuesto mediano**.

Esto ayuda a reducir:

- nubes residuales;
- valores extremos;
- pequeñas diferencias entre fechas.

El procedimiento será:

Sentinel-2

↓

filtrar área

↓

filtrar período

↓

filtrar escenas muy nubladas

↓

enmascarar nubes

↓

calcular mediana

In [5]:
# ============================================================
# Definir período
# ============================================================

START_DATE = "2025-07-01"
END_DATE   = "2025-09-01"

print(
    "Período:",
    START_DATE,
    "a",
    END_DATE
)

Período: 2025-07-01 a 2025-09-01


In [6]:
# ============================================================
# Cargar Sentinel-2
# ============================================================

s2 = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(roi)
    .filterDate(
        START_DATE,
        END_DATE
    )
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            20
        )
    )
)

print(
    "Número de imágenes:",
    s2.size().getInfo()
)

Número de imágenes: 22


## Máscara sencilla de nubes

Sentinel-2 incluye una banda denominada `SCL`
(Scene Classification Layer).

Esta banda clasifica diferentes tipos de píxeles.

Para mantener el ejercicio sencillo eliminaremos solamente:

- píxeles saturados;
- sombras de nubes;
- nubes;
- cirrus;
- nieve/hielo.

Esta es una máscara simplificada apropiada para el ejercicio.

En aplicaciones operativas pueden utilizarse procedimientos de
enmascaramiento atmosférico más avanzados.

In [7]:
# ============================================================
# Función sencilla para eliminar nubes
# ============================================================

def mask_clouds(image):

    scl = image.select("SCL")

    clear = (
        scl.neq(1)   # saturado
        .And(scl.neq(3))   # sombra
        .And(scl.neq(8))   # nube media
        .And(scl.neq(9))   # nube alta
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # nieve/hielo
    )

    return image.updateMask(clear)

In [8]:
# ============================================================
# Crear compuesto Sentinel-2
# ============================================================

s2_composite = (
    s2
    .map(mask_clouds)
    .median()
    .select([
        "B2",
        "B3",
        "B4",
        "B5",
        "B8",
        "B11"
    ])
    .multiply(0.0001)
    .clip(roi)
)

print(
    "Compuesto Sentinel-2 preparado."
)

Compuesto Sentinel-2 preparado.


In [9]:
# ============================================================
# Visualización en color natural
# ============================================================

rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.0,
    "max": 0.25,
    "gamma": 1.2
}

Map_rgb = geemap.Map()

Map_rgb.centerObject(
    roi,
    10
)

Map_rgb.addLayer(
    s2_composite,
    rgb_vis,
    "Sentinel-2 RGB"
)

Map_rgb

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Primero observa la imagen

Antes de calcular cualquier índice:

1. observa el color del agua;
2. identifica zonas claras y oscuras;
3. observa los sectores próximos a Cohana;
4. compara la bahía con zonas abiertas.

### Preguntas

> ¿El color del agua es homogéneo en toda el área?

> ¿Puedes reconocer visualmente sectores con agua más verdosa o turbia?

No interpretes todavía estos colores como concentraciones.

Primero calcularemos algunos indicadores espectrales.

# 7. Máscara de agua: MNDWI

Antes de analizar calidad del agua necesitamos distinguir aproximadamente
los píxeles de agua de los píxeles terrestres.

Utilizaremos el:

**Modified Normalized Difference Water Index (MNDWI)**

\[
MNDWI =
\frac{Green-SWIR}
{Green+SWIR}
\]

Para Sentinel-2:

\[
MNDWI =
\frac{B3-B11}
{B3+B11}
\]

En este ejercicio utilizaremos:

**MNDWI > 0**

como una máscara sencilla de agua.

In [10]:
# ============================================================
# Calcular MNDWI
# ============================================================

mndwi = (
    s2_composite
    .normalizedDifference([
        "B3",
        "B11"
    ])
    .rename("MNDWI")
)

water_mask = mndwi.gt(0)

print("MNDWI calculado.")

MNDWI calculado.


In [11]:
# ============================================================
# Visualizar MNDWI
# ============================================================

mndwi_vis = {
    "min": -0.5,
    "max": 0.7,
    "palette": [
        "8c510a",
        "f6e8c3",
        "ffffff",
        "67a9cf",
        "2166ac"
    ]
}

Map_water = geemap.Map()

Map_water.centerObject(
    roi,
    10
)

Map_water.addLayer(
    mndwi,
    mndwi_vis,
    "MNDWI"
)

Map_water.addLayer(
    water_mask.selfMask(),
    {"palette": ["0000ff"]},
    "Máscara de agua",
    False
)

Map_water

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Pregunta

Activa y desactiva la máscara de agua.

> ¿El MNDWI identifica correctamente la mayor parte del lago?

Observa especialmente:

- zonas costeras;
- vegetación acuática;
- pequeñas bahías.

La máscara no es perfecta.

Su propósito es simplemente evitar que los índices posteriores
incluyan grandes superficies terrestres.

# 8. Proxy de clorofila-a: NDCI

La clorofila-a está asociada a la presencia de fitoplancton.

Utilizaremos el:

**Normalized Difference Chlorophyll Index (NDCI)**

\[
NDCI =
\frac{RedEdge-Red}
{RedEdge+Red}
\]

Para Sentinel-2:

\[
NDCI =
\frac{B5-B4}
{B5+B4}
\]

donde:

- B4 = rojo (~665 nm);
- B5 = red-edge (~704 nm).

El NDCI aprovecha cambios espectrales asociados con absorción de
clorofila en el rojo y aumento de reflectancia alrededor del red-edge.

---

> **Importante**

En este ejercicio NDCI será utilizado como un **proxy relativo**.

Un valor alto de NDCI NO significa directamente una concentración
determinada de clorofila-a en mg/m³.

Para obtener concentraciones sería necesaria una calibración o un
algoritmo validado para el sistema estudiado.

In [13]:
# ============================================================
# Calcular NDCI
# ============================================================

ndci = (
    s2_composite
    .normalizedDifference([
        "B5",
        "B4"
    ])
    .rename("NDCI")
    .updateMask(water_mask)
)

print("NDCI calculado.")

NDCI calculado.


In [14]:
# ============================================================
# Visualizar NDCI
# ============================================================

ndci_vis = {
    "min": -0.2,
    "max": 0.4,
    "palette": [
        "440154",
        "3b528b",
        "21918c",
        "5ec962",
        "fde725"
    ]
}

Map_ndci = geemap.Map()

Map_ndci.centerObject(
    roi,
    10
)

Map_ndci.addLayer(
    ndci,
    ndci_vis,
    "NDCI"
)

Map_ndci

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Interpretación

Observa los patrones espaciales de NDCI.

### Preguntas

1. ¿Los valores son homogéneos en todo el lago?

2. ¿Dónde aparecen los valores relativamente más altos?

3. ¿Se observan diferencias entre Bahía de Cohana y aguas más abiertas?

4. ¿Coinciden las zonas de NDCI alto con las diferencias de color
   observadas previamente en la imagen RGB?

Por ahora interpreta únicamente diferencias **relativas**:

NDCI más bajo → menor señal relativa

NDCI más alto → mayor señal relativa asociada a clorofila

# 9. Proxy de turbidez: NDTI

La turbidez modifica la cantidad de luz que es dispersada por el agua.

Una forma sencilla de explorar esta señal es mediante el:

**Normalized Difference Turbidity Index (NDTI)**

\[
NDTI =
\frac{Red-Green}
{Red+Green}
\]

Para Sentinel-2:

\[
NDTI =
\frac{B4-B3}
{B4+B3}
\]

En aguas con mayor presencia de partículas, la reflectancia en el rojo
puede aumentar con relación a aguas más claras.

---

> Nuevamente, NDTI será utilizado como un indicador relativo.

No interpretaremos sus valores directamente como unidades de turbidez
como NTU.

In [15]:
# ============================================================
# Calcular NDTI
# ============================================================

ndti = (
    s2_composite
    .normalizedDifference([
        "B4",
        "B3"
    ])
    .rename("NDTI")
    .updateMask(water_mask)
)

print("NDTI calculado.")

NDTI calculado.


In [16]:
# ============================================================
# Visualizar NDTI
# ============================================================

ndti_vis = {
    "min": -0.2,
    "max": 0.2,
    "palette": [
        "313695",
        "74add1",
        "ffffbf",
        "f46d43",
        "a50026"
    ]
}

Map_ndti = geemap.Map()

Map_ndti.centerObject(
    roi,
    10
)

Map_ndti.addLayer(
    ndti,
    ndti_vis,
    "NDTI"
)

Map_ndti

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Preguntas

1. ¿Dónde aparecen los valores más altos de NDTI?

2. ¿Se concentran cerca de la costa?

3. ¿Existen gradientes desde las bahías hacia aguas abiertas?

4. Compara mentalmente NDTI con NDCI.

> ¿Los hotspots de turbidez coinciden necesariamente con los
> hotspots asociados a clorofila?

No necesariamente.

Fitoplancton y partículas minerales pueden modificar la señal óptica
del agua de maneras diferentes.

# 10. Proxy de sólidos suspendidos

Los sólidos suspendidos incrementan la dispersión de la luz y pueden
producir aumentos importantes de reflectancia en las bandas visibles,
especialmente en el rojo.

Para mantener este ejercicio sencillo no aplicaremos un algoritmo
empírico de TSS.

Utilizaremos directamente la reflectancia de:

**Sentinel-2 B4 (~665 nm)**

como un indicador relativo.

Por tanto:

\[
TSS_{proxy} \sim Reflectancia_{rojo}
\]

Esto NO significa que la reflectancia sea equivalente a una
concentración de sólidos suspendidos en mg/L.

Solamente compararemos patrones espaciales.

In [17]:
# ============================================================
# Reflectancia roja como proxy de sólidos suspendidos
# ============================================================

red_proxy = (
    s2_composite
    .select("B4")
    .rename("Red_reflectance")
    .updateMask(water_mask)
)

print(
    "Proxy de sólidos suspendidos preparado."
)

Proxy de sólidos suspendidos preparado.


In [18]:
# ============================================================
# Visualizar reflectancia roja
# ============================================================

red_vis = {
    "min": 0.0,
    "max": 0.08,
    "palette": [
        "001a4d",
        "0066cc",
        "66ccff",
        "ffff66",
        "ff9900",
        "cc0000"
    ]
}

Map_red = geemap.Map()

Map_red.centerObject(
    roi,
    10
)

Map_red.addLayer(
    red_proxy,
    red_vis,
    "Reflectancia roja B4"
)

Map_red

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Preguntas

Compara el mapa de B4 con el mapa de NDTI.

1. ¿Los patrones son similares?

2. ¿Dónde encontramos mayor reflectancia roja?

3. ¿Qué podría explicar una señal elevada cerca de desembocaduras
   o sectores costeros?

---

### Una distinción importante

**Turbidez ≠ sólidos suspendidos**

La turbidez es una propiedad óptica relacionada con la dispersión
de la luz.

Los sólidos suspendidos representan una cantidad de material presente
en el agua.

Ambas variables están relacionadas, pero no son equivalentes.

In [19]:
# ============================================================
# Comparar productos Sentinel-2
# ============================================================

Map_quality = geemap.Map()

Map_quality.centerObject(
    roi,
    10
)

Map_quality.addLayer(
    s2_composite,
    rgb_vis,
    "RGB",
    True
)

Map_quality.addLayer(
    ndci,
    ndci_vis,
    "NDCI - Clorofila proxy",
    False
)

Map_quality.addLayer(
    ndti,
    ndti_vis,
    "NDTI - Turbidez proxy",
    False
)

Map_quality.addLayer(
    red_proxy,
    red_vis,
    "B4 - Sólidos suspendidos proxy",
    False
)

Map_quality

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Detente unos minutos en este mapa

Activa y desactiva:

- RGB;
- NDCI;
- NDTI;
- B4.

### Pregunta central

> ¿Los tres indicadores muestran exactamente el mismo patrón espacial?

Si no lo hacen:

> ¿qué procesos diferentes podrían estar representando?

# 12. Productos derivados MODIS Aqua

Hasta ahora seguimos esta secuencia:

**bandas Sentinel-2**

↓

**índices espectrales**

↓

**proxies relativos**

Ahora utilizaremos productos que ya han sido procesados mediante
algoritmos bio-ópticos y térmicos.

Utilizaremos:

### Clorofila-a

`chlor_a`

Unidad:

**mg/m³**

### Temperatura superficial

`sst`

Unidad:

**°C**

Utilizaremos exactamente el mismo período empleado para Sentinel-2:

**julio–agosto de 2025**

y calcularemos la mediana temporal.

In [20]:
# ============================================================
# Cargar MODIS Aqua
# ============================================================

modis = (
    ee.ImageCollection(
        "NASA/OCEANDATA/MODIS-Aqua/L3SMI"
    )
    .filterDate(
        START_DATE,
        END_DATE
    )
)

print(
    "Número de imágenes MODIS:",
    modis.size().getInfo()
)

Número de imágenes MODIS: 62


In [21]:
# ============================================================
# Crear compuestos MODIS
# ============================================================

modis_chl = (
    modis
    .select("chlor_a")
    .median()
    .clip(roi)
)

modis_sst = (
    modis
    .select("sst")
    .median()
    .clip(roi)
)

print(
    "Compuestos MODIS preparados."
)

Compuestos MODIS preparados.


In [22]:
# ============================================================
# Visualizar clorofila-a MODIS
# ============================================================

chl_vis = {
    "min": 0,
    "max": 5,
    "palette": [
        "440154",
        "3b528b",
        "21918c",
        "5ec962",
        "fde725"
    ]
}

Map_chl = geemap.Map()

Map_chl.centerObject(
    roi,
    9
)

Map_chl.addLayer(
    modis_chl,
    chl_vis,
    "MODIS chlor_a"
)

Map_chl

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Observa la diferencia de escala

Recuerda:

Sentinel-2:

**10–20 m**

MODIS L3SMI:

**~4.6 km**

### Preguntas

1. ¿Cuánto detalle espacial se perdió?

2. ¿Puedes distinguir pequeñas bahías?

3. ¿Es posible identificar detalles costeros como con Sentinel-2?

4. ¿Qué ventaja tiene entonces MODIS?

Una ventaja importante es su alta frecuencia temporal y la disponibilidad
directa de productos derivados.

In [23]:
# ============================================================
# Visualizar temperatura superficial MODIS
# ============================================================

sst_vis = {
    "min": 5,
    "max": 20,
    "palette": [
        "313695",
        "4575b4",
        "74add1",
        "fee090",
        "f46d43",
        "a50026"
    ]
}

Map_sst = geemap.Map()

Map_sst.centerObject(
    roi,
    9
)

Map_sst.addLayer(
    modis_sst,
    sst_vis,
    "MODIS SST"
)

Map_sst

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Preguntas

1. ¿La temperatura es homogénea en todo el sector?

2. ¿Se observan diferencias entre zonas costeras y aguas abiertas?

3. ¿Qué procesos podrían producir diferencias de temperatura?

Por ejemplo:

- profundidad;
- mezcla del agua;
- radiación solar;
- viento;
- intercambio con tributarios.

---

### Una diferencia importante

Sentinel-2 no posee una banda térmica.

Por tanto:

**Sentinel-2 → gran detalle espacial en información óptica**

**MODIS → permite incorporar temperatura superficial**

# 13. ¿Cuál sensor es mejor?

La respuesta es:

> **depende de la pregunta.**

Los dos sensores proporcionan información complementaria.

| Característica | Sentinel-2 | MODIS Aqua |
|---|---|---|
| Resolución espacial | 10–20 m | ~4.6 km |
| Frecuencia temporal | Menor | Alta |
| Clorofila | NDCI proxy | chlor_a |
| Turbidez | NDTI proxy | No utilizada aquí |
| Sólidos suspendidos | B4 proxy | No utilizado aquí |
| Temperatura | No disponible | SST |
| Detalle costero | Alto | Bajo |
| Análisis regional | Bueno | Muy bueno |

In [24]:
# ============================================================
# Comparar Sentinel-2 NDCI y MODIS chlor_a
# ============================================================

Map_compare = geemap.Map()

Map_compare.centerObject(
    roi,
    9
)

Map_compare.addLayer(
    ndci,
    ndci_vis,
    "Sentinel-2 NDCI",
    True
)

Map_compare.addLayer(
    modis_chl,
    chl_vis,
    "MODIS chlor_a",
    False
)

Map_compare

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Pregunta

Activa alternativamente:

- Sentinel-2 NDCI;
- MODIS chlor_a.

No compares directamente los valores numéricos porque representan
variables diferentes.

En cambio, compara:

**los patrones espaciales generales.**

> ¿Ambos productos sugieren diferencias entre sectores costeros y
> aguas abiertas?

> ¿Qué información desaparece al pasar de Sentinel-2 a MODIS?

# 14. Comparación de tres puntos

Finalizaremos el análisis Sentinel-2 con una comparación sencilla.

Seleccionaremos tres puntos aproximadamente alineados desde
Bahía de Cohana hacia aguas más abiertas:

- P1: interior de Bahía de Cohana;
- P2: zona intermedia;
- P3: aguas más abiertas.

Extraeremos:

- NDCI;
- NDTI;
- reflectancia roja.

El objetivo no es realizar un análisis estadístico.

Queremos simplemente comprobar si existe un **gradiente espacial**.

In [25]:
# ============================================================
# Definir tres puntos
# ============================================================

points = ee.FeatureCollection([

    ee.Feature(
        ee.Geometry.Point([
            -68.675,
            -16.375
        ]),
        {"site": "P1 - Cohana"}
    ),

    ee.Feature(
        ee.Geometry.Point([
            -68.740,
            -16.395
        ]),
        {"site": "P2 - Intermedio"}
    ),

    ee.Feature(
        ee.Geometry.Point([
            -68.820,
            -16.420
        ]),
        {"site": "P3 - Lago abierto"}
    )

])

print("Puntos definidos.")

Puntos definidos.


In [26]:
# ============================================================
# Visualizar puntos
# ============================================================

Map_points = geemap.Map()

Map_points.centerObject(
    roi,
    10
)

Map_points.addLayer(
    s2_composite,
    rgb_vis,
    "RGB"
)

Map_points.addLayer(
    points,
    {"color": "red"},
    "Puntos"
)

Map_points

Map(center=[-16.37501993392347, -68.72499999999968], controls=(WidgetControl(options=['position', 'transparent…

### Antes de continuar

Comprueba visualmente que los tres puntos se encuentran sobre agua.

Esta verificación es importante siempre que trabajemos con puntos
de muestreo en imágenes satelitales.

In [27]:
# ============================================================
# Combinar los tres indicadores
# ============================================================

quality_indices = (
    ndci
    .addBands(ndti)
    .addBands(red_proxy)
)

print(
    quality_indices.bandNames().getInfo()
)

['NDCI', 'NDTI', 'Red_reflectance']


In [28]:
# ============================================================
# Extraer valores en los tres puntos
# ============================================================

samples = quality_indices.sampleRegions(
    collection=points,
    properties=["site"],
    scale=20,
    geometries=False
)

samples_info = samples.getInfo()

records = [
    feature["properties"]
    for feature
    in samples_info["features"]
]

df_points = pd.DataFrame(records)

display(df_points)

,NDCI,NDTI,Red_reflectance,site
0,0.156522,-0.361842,0.0097,P3 - Lago abierto


In [29]:
# ============================================================
# Ordenar tabla
# ============================================================

df_points = df_points[
    [
        "site",
        "NDCI",
        "NDTI",
        "Red_reflectance"
    ]
]

display(
    df_points.round(4)
)

,site,NDCI,NDTI,Red_reflectance
0,P3 - Lago abierto,0.1565,-0.3618,0.0097


### Interpreta la tabla

Compara P1, P2 y P3.

### Preguntas

1. ¿NDCI disminuye desde Cohana hacia aguas abiertas?

2. ¿NDTI presenta el mismo comportamiento?

3. ¿Qué ocurre con la reflectancia roja?

4. ¿Los tres indicadores muestran un gradiente similar?

5. ¿Qué podría explicar las diferencias?

---

Recuerda:

estos valores representan únicamente las condiciones del compuesto
satelital utilizado en el ejercicio.

No debemos generalizar automáticamente estos patrones a todas las
épocas del año.

# 15. Interpretación conjunta

Hemos analizado el Lago Menor utilizando seis perspectivas:

**RGB**

↓

**MNDWI**

↓

**NDCI**

↓

**NDTI**

↓

**Reflectancia roja**

↓

**MODIS chlor_a + SST**

---

## Pregunta 1

¿Dónde aparecen los principales hotspots en los productos Sentinel-2?

---

## Pregunta 2

¿NDCI, NDTI y B4 muestran exactamente el mismo patrón?

¿Qué podría explicar sus diferencias?

---

## Pregunta 3

¿Existe evidencia de un gradiente desde Bahía de Cohana hacia
aguas abiertas?

---

## Pregunta 4

¿Los patrones generales de NDCI y clorofila MODIS parecen consistentes?

---

## Pregunta 5

¿Qué información aporta la temperatura superficial que no podemos
obtener con Sentinel-2?

---

## Pregunta 6

Si tuvieras que estudiar una pequeña bahía:

> ¿qué sensor elegirías?

Si quisieras estudiar variabilidad temporal frecuente a escala regional:

> ¿qué sensor podría ser más conveniente?

---

## Pregunta 7

¿Podemos afirmar solamente con estos resultados que una zona está
"contaminada"?

¿Por qué no?

# 16. Limitaciones

Los productos satelitales son herramientas muy poderosas para monitorear
la calidad superficial del agua, pero presentan varias limitaciones.

## 1. Proxies no equivalen a concentraciones

En este ejercicio:

- NDCI no equivale directamente a mg/m³ de clorofila-a;
- NDTI no equivale directamente a NTU;
- B4 no equivale directamente a mg/L de sólidos suspendidos.

Para convertir estos proxies en concentraciones se requiere calibración
con observaciones de campo y algoritmos apropiados.

---

## 2. Aguas ópticamente complejas

Clorofila, sedimentos, materia orgánica y vegetación acuática pueden
modificar simultáneamente la señal espectral.

Esto dificulta separar cada componente.

---

## 3. Efectos costeros

Píxeles próximos a la costa pueden contener simultáneamente:

- agua;
- vegetación;
- suelo.

Esto puede modificar la señal observada.

---

## 4. Nubes y atmósfera

Las observaciones ópticas dependen de condiciones atmosféricas adecuadas.

El procedimiento de nubes utilizado en este ejercicio es deliberadamente
sencillo.

---

## 5. Resolución espacial

MODIS tiene una resolución mucho menor que Sentinel-2.

Un píxel MODIS puede cubrir varios kilómetros y mezclar diferentes
condiciones de agua.

---

## 6. Los productos derivados también tienen incertidumbre

Aunque `chlor_a` y `sst` de MODIS vienen expresados en unidades físicas,
siguen siendo estimaciones obtenidas mediante algoritmos satelitales.

Por esta razón, siempre que sea posible deben compararse con
observaciones in situ.

# 17. Ejercicio adicional

Ahora aplica el procedimiento a otra zona o período.

Puedes modificar:

### Opción A — Área

Cambia el `roi` y analiza otro sector del Lago Titicaca.

### Opción B — Fecha

Cambia:

`START_DATE`

y

`END_DATE`

y repite el ejercicio en otra época del año.

### Opción C — Comparación estacional

Compara un período de estación seca con un período de estación húmeda.

---

Para cada caso:

1. observa RGB;
2. calcula MNDWI;
3. calcula NDCI;
4. calcula NDTI;
5. observa B4;
6. identifica posibles hotspots;
7. interpreta las diferencias.

---

## Pregunta final

> ¿Los patrones de calidad del agua permanecen estables o cambian
> dependiendo del lugar y del período observado?

# 18. Referencias

### Sentinel-2

European Space Agency / Copernicus.
Sentinel-2 MSI Level-2A Surface Reflectance.

Google Earth Engine Data Catalog:
`COPERNICUS/S2_SR_HARMONIZED`

---

### MODIS Aqua

NASA Ocean Biology Processing Group.
MODIS-Aqua Level-3 Standard Mapped Image.

Google Earth Engine Data Catalog:
`NASA/OCEANDATA/MODIS-Aqua/L3SMI`

---

### NDCI

Mishra, S. & Mishra, D. R. (2012).

Normalized difference chlorophyll index:
A novel model for remote estimation of chlorophyll-a
concentration in turbid productive waters.

Remote Sensing of Environment, 117, 394–406.

DOI: 10.1016/j.rse.2011.10.016

---

### Lago Titicaca y MODIS

Xu, H., Velaochaga, G., Siguayro, H., Paulino, C.,
Alburqueque, E., Escudero, L., Atiquipa, J. & Gamarra, C. (2021).

Validación de datos satelitales de la concentración de clorofila-a,
temperatura superficial y coeficiente de atenuación difusa
en el Lago Titicaca.

Instituto del Mar del Perú – IMARPE.

---

## Mensaje final

Las observaciones satelitales permiten estudiar patrones espaciales
y temporales de algunas propiedades relacionadas con la calidad del agua.

Sin embargo:

> **el satélite complementa el monitoreo in situ; no necesariamente
> lo reemplaza.**

La interpretación adecuada requiere comprender:

- qué mide el sensor;
- qué representa cada índice;
- qué algoritmo genera cada producto;
- cuál es su resolución;
- cuáles son sus incertidumbres.